# VibeML: Personalized Spotify Playlist Agent

This notebook defines the VibeML agent, a personalized Spotify playlist generation agent that combines a user's listening history with real-time mood and activity input to build a playlist for the current moment.

##Install Dependencies

In [0]:
# Install dependencies
%pip install spotipy

In [0]:
%restart_python

## Imports and Setup

Import all necessary libraries and set up the environment for the agent.

In [0]:
# Core libraries
import os
import json
import pandas as pd

# Spotify API
import spotipy
from spotipy.oauth2 import SpotifyOAuth

# Databricks / MLflow
import mlflow

# LLM - we will fill this in once we confirm which LLM we are using

### Spotify Authentication

This section sets up the Spotify client using the full OAuth flow, which gives the agent access to user specific data including liked songs, playlists, listening history, and the ability to save playlists.

#### Setup Steps (first time only)

1. Go to developer.spotify.com and log in with your Spotify account
2. Click "Create app" and fill in the following:
   - App name: VibeML
   - App description: Personalized Spotify Playlist Agent
   - Redirect URI: http://127.0.0.1:8080
   - Check the "Web API" box
3. Once the app is created click "Settings" and copy your Client ID and Client Secret
4. Run the credentials cell below and enter your Client ID and Client Secret when prompted
5. Run this cell, it will print a Spotify authorization URL in the output:
    e.g. with the client ID and key as your specific credentials : "https://accounts.spotify.com/authorize?client_id=e883018d172a497a94c2e78f9b300117&response_type=code&redirect_uri=http%3A%2F%2F127.0.0.1%3A8080&scope=user-library-read+user-read-recently-played+playlist-read-private+playlist-modify-public+playlist-modify-private+user-read-currently-playing"
    
6. Copy that URL and paste it into your browser
7. Log in with your Spotify account and click "Agree"
8. You will be redirected to a URL starting with http://127.0.0.1:8080/?code=
9. Copy that full URL from your browser address bar
10. Paste it into the notebook where it says "Enter the URL you were redirected to:" and hit enter
11. You should see "Spotify connected successfully" with your display name

In [0]:
# CREDENTIALS CELL
import os

# Set Spotify credentials manually at runtime
# These are never stored in the notebook or GitHub
os.environ["SPOTIFY_CLIENT_ID"] = dbutils.widgets.get("client_id") if False else input("Enter your Spotify Client ID: ")
os.environ["SPOTIFY_CLIENT_SECRET"] = dbutils.widgets.get("client_secret") if False else input("Enter your Spotify Client Secret: ")
os.environ["SPOTIFY_REDIRECT_URI"] = "http://127.0.0.1:8080"

In [0]:
# AUTHENTICATION CELL
from spotipy.oauth2 import SpotifyClientCredentials

# Set up the Spotify client using client credentials
sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
    client_id=os.environ["SPOTIFY_CLIENT_ID"],
    client_secret=os.environ["SPOTIFY_CLIENT_SECRET"]
))

In [0]:
# Test the connection
results = sp.search(q="test", limit=1)
print("Spotify connected successfully")
print(f"Test track: {results['tracks']['items'][0]['name']}")

In [0]:
from spotipy.oauth2 import SpotifyOAuth

# Define the scopes we need from the Spotify API
SPOTIFY_SCOPES = " ".join([
    "user-library-read",
    "user-read-recently-played",
    "playlist-read-private",
    "playlist-modify-public",
    "playlist-modify-private",
    "user-read-currently-playing"
])

# Set up the Spotify OAuth client
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    client_id=os.environ["SPOTIFY_CLIENT_ID"],
    client_secret=os.environ["SPOTIFY_CLIENT_SECRET"],
    redirect_uri=os.environ["SPOTIFY_REDIRECT_URI"],
    scope=SPOTIFY_SCOPES,
    open_browser=False
))

# Test the connection
user = sp.current_user()
print(f"Spotify connected successfully")
print(f"Logged in as: {user['display_name']}")

## LLM Setup

This section configures the LLM that powers the agent's reasoning

In [0]:
# System prompt that tells the LLM what its job is
SYSTEM_PROMPT = """
You are VibeML, a personalized Spotify playlist agent. Your job is to help the user build a playlist that fits exactly what they are feeling or doing right now.

You have access to the following tools:
- get_user_profile: Pull the user's liked songs, playlists, and listening history from Spotify
- get_currently_playing: Fetch the track the user is currently listening to
- search_catalog: Verify a song exists and is available on Spotify
- search_songs: Search for songs by audio feature ranges like energy, valence, tempo, and danceability
- trend_filter: Filter songs by release era or artist popularity
- vibe_mapping: Translate the user's mood or activity description into target audio feature ranges
- feedback_refinement: Adjust audio feature ranges based on the user's tester song feedback
- save_playlist: Save the final playlist to the user's Spotify account

Follow these steps when helping a user:
1. Pull the user's profile to understand their taste
2. Ask about their current mood, activity, and desired vibe
3. Use vibe_mapping to translate their input into audio feature ranges
4. Search for matching songs using search_songs and trend_filter as needed
5. Present 3 tester songs and collect feedback
6. Use feedback_refinement to adjust if needed
7. Build and save the final playlist

Only help with music playlist requests. Politely decline anything outside of that scope.
"""

In [0]:
from openai import OpenAI
import mlflow

# LLM model configuration
LLM_MODEL = "databricks-gpt-oss-120b" # Can be changed, just an example

# Get the workspace URL and token automatically from the Databricks environment
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
DATABRICKS_HOST = spark.conf.get("spark.databricks.workspaceUrl")

# Set up the Databricks OpenAI compatible client
client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=f"https://{DATABRICKS_HOST}/serving-endpoints"
)

# Test that the model endpoint is accessible
def test_llm_setup():
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "user", "content": "Say hello in one sentence."}
        ]
    )
    # Extract text content only, skipping reasoning blocks
    content = response.choices[0].message.content
    if isinstance(content, list):
        text = " ".join([block["text"] for block in content if block.get("type") == "text"])
    else:
        text = content
    
    print("LLM setup successful")
    print(f"Model: {LLM_MODEL}")
    print(f"Response: {text}")

test_llm_setup()

## Tool Definitions

The following functions represent the tools the agent can call during its reasoning process.

### Tool 1 - Get User Profile: get_user_profile
- Pulls the user's liked songs, saved playlists, and listening history from the Spotify API to build a baseline understanding of their musical taste.

In [0]:
def get_user_profile(sp):
    """
    Pulls the user's liked songs, saved playlists, and listening history from the Spotify API.
    
    Args:
        sp: Authenticated Spotipy client
    Returns:
        dict: User's liked songs, playlists, and listening history
    """
    # Get liked songs (up to 50)
    liked_songs = sp.current_user_saved_tracks(limit=50)
    liked_tracks = [
        {
            "track_name": item["track"]["name"],
            "artist": item["track"]["artists"][0]["name"],
            "track_id": item["track"]["id"]
        }
        for item in liked_songs["items"]
    ]

    # Get user playlists (up to 20)
    playlists = sp.current_user_playlists(limit=20)
    user_playlists = [
        {
            "playlist_name": playlist["name"],
            "track_count": playlist.get("tracks", {}).get("total", 0)
        }
        for playlist in playlists["items"]
    ]

    # Get recently played tracks (up to 20)
    recently_played = sp.current_user_recently_played(limit=20)
    recent_tracks = [
        {
            "track_name": item["track"]["name"],
            "artist": item["track"]["artists"][0]["name"],
            "track_id": item["track"]["id"]
        }
        for item in recently_played["items"]
    ]

    return {
        "liked_tracks": liked_tracks,
        "playlists": user_playlists,
        "recently_played": recent_tracks
    }

In [0]:
# Test the tool
profile = get_user_profile(sp)
print(f"Liked songs pulled: {len(profile['liked_tracks'])}")
print(f"Playlists pulled: {len(profile['playlists'])}")
print(f"Recently played pulled: {len(profile['recently_played'])}")
print("\nFirst 3 liked songs:")
for track in profile["liked_tracks"][:3]:
    print(f"  {track['track_name']} by {track['artist']}")
print("\nFirst 3 recently played:")
for track in profile["recently_played"][:3]:
    print(f"  {track['track_name']} by {track['artist']}")

### Tool 2 - Get Current Song Playing: get_currently_playing
- Fetches the track the user is currently listening to from the Spotify API.


In [0]:
def get_currently_playing(sp):
    """
    Fetches the track the user is currently listening to from the Spotify API.
    
    Args:
        sp: Authenticated Spotipy client
    Returns:
        dict: Currently playing track info
    """
    pass

### Tool 3 - Search Catalog: search_catalog
- Searches the Spotify catalog by track name or artist to verify that a recommended song exists on the platform and is available to the user.

In [0]:
def search_catalog(sp, track_name=None, artist_name=None):
    """
    Searches the Spotify catalog to verify a track exists and is available.
    
    Args:
        sp: Authenticated Spotipy client
        track_name (str): Name of the track to search for
        artist_name (str): Name of the artist
    Returns:
        dict: Track info if found, None if not found
    """
    query = ""
    if track_name:
        query += f"track:{track_name} "
    if artist_name:
        query += f"artist:{artist_name}"
    
    query = query.strip()
    
    if not query:
        print("No search parameters provided")
        return None
    
    results = sp.search(q=query, limit=1, type="track")
    
    if results["tracks"]["items"]:
        track = results["tracks"]["items"][0]
        return {
            "track_name": track["name"],
            "artist": track["artists"][0]["name"],
            "track_id": track["id"],
            "album": track["album"]["name"],
            "available": True
        }
    else:
        return None

In [0]:
# Test the tool
result = search_catalog(sp, track_name="Rap God", artist_name="Eminem")
if result:
    print(f"Track found: {result['track_name']} by {result['artist']}")
    print(f"Album: {result['album']}")
    print(f"Track ID: {result['track_id']}")
else:
    print("Track not found")

### Tool 4 - Song Search: search_songs
- Queries the processed Kaggle dataset by audio feature ranges to find matching tracks.

In [0]:
def search_songs(energy=None, valence=None, tempo=None, danceability=None, speechiness=None, instrumentalness=None, limit=20):
    """
    Queries the Spotify Tracks Dataset (114,000 tracks across 125 genres) by audio 
    feature ranges to find matching tracks.
    
    Args:
        energy (tuple): Min and max energy range e.g. (0.7, 1.0)
        valence (tuple): Min and max valence range
        tempo (tuple): Min and max tempo range
        danceability (tuple): Min and max danceability range
        speechiness (tuple): Min and max speechiness range
        instrumentalness (tuple): Min and max instrumentalness range
        limit (int): Max number of tracks to return
    Returns:
        DataFrame: Tracks matching the specified audio feature ranges
    """
    df = spark.table("main.vibeml.vibeml_tracks_processed").toPandas()
    
    if energy:
        df = df[(df["energy"] >= energy[0]) & (df["energy"] <= energy[1])]
    if valence:
        df = df[(df["valence"] >= valence[0]) & (df["valence"] <= valence[1])]
    if tempo:
        df = df[(df["tempo"] >= tempo[0]) & (df["tempo"] <= tempo[1])]
    if danceability:
        df = df[(df["danceability"] >= danceability[0]) & (df["danceability"] <= danceability[1])]
    if speechiness:
        df = df[(df["speechiness"] >= speechiness[0]) & (df["speechiness"] <= speechiness[1])]
    if instrumentalness:
        df = df[(df["instrumentalness"] >= instrumentalness[0]) & (df["instrumentalness"] <= instrumentalness[1])]
    
    return df.sample(min(limit, len(df))).reset_index(drop=True)

In [0]:
# Test the tool
results = search_songs(energy=(0.8, 1.0), danceability=(0.7, 1.0))
print(f"Found {len(results)} tracks")
print(results[["track_name", "artists", "energy", "danceability"]].head(5))

### Tool 5 - Trend Filter: trend_filter
- Queries the second Kaggle dataset to narrow recommendations by release era or artist popularity.


In [0]:
def trend_filter(era=None, popularity=None):
    """
    Queries the Spotify Global Music Dataset (2009-2025) to narrow recommendations 
    by release era or artist popularity.
    
    Args:
        era (tuple): Min and max release year range e.g. (2009, 2015)
        popularity (tuple): Min and max artist popularity range
    Returns:
        DataFrame: Tracks matching the specified trend filters
    """
    pass

### Tool 6 - Vibe Mapping: vibe_mapping
- Takes the user's natural language mood or activity description and uses the LLM to translate it into target audio feature ranges.

In [0]:
def vibe_mapping(user_input):
    """
    Takes the user's natural language description of their mood or activity and passes it to the LLM,
    which interprets the input and outputs a set of target audio feature ranges in a structured format.
    
    Args:
        user_input (str): User's natural language description of their mood or activity
    Returns:
        dict: Target audio feature ranges derived from the user's input
    """
    prompt = f"""
    You are a music expert. Based on the user's mood or activity description, return a JSON object 
    with target audio feature ranges for a Spotify playlist. 
    
    Each feature should have a "min" and "max" value between 0.0 and 1.0, except tempo which is in BPM.
    
    Features to include:
    - energy (0.0 to 1.0)
    - valence (0.0 to 1.0)
    - danceability (0.0 to 1.0)
    - speechiness (0.0 to 1.0)
    - instrumentalness (0.0 to 1.0)
    - tempo (60 to 200 BPM)
    
    User description: {user_input}
    
    Return ONLY a valid JSON object with no explanation or extra text. Example format:
    {{
        "energy": {{"min": 0.7, "max": 1.0}},
        "valence": {{"min": 0.5, "max": 1.0}},
        "danceability": {{"min": 0.7, "max": 1.0}},
        "speechiness": {{"min": 0.0, "max": 0.1}},
        "instrumentalness": {{"min": 0.0, "max": 0.3}},
        "tempo": {{"min": 120, "max": 180}}
    }}
    """
    
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    
    # Handle both string and list response formats
    content = response.choices[0].message.content
    if isinstance(content, list):
        raw = " ".join([block["text"] for block in content if block.get("type") == "text"])
    else:
        raw = content
    
    # Clean up any markdown formatting if present
    raw = raw.replace("```json", "").replace("```", "").strip()
    
    return json.loads(raw)

In [0]:
# Test the tool
result = vibe_mapping("I am heading to the gym and want something high energy")
print("Vibe mapping result:")
print(json.dumps(result, indent=2))

### Tool 7 - Feedback Refinement: feedback_refinement
- Adjusts audio feature target ranges based on the user's tester song feedback.



In [0]:
def feedback_refinement(accepted_tracks, rejected_tracks, current_ranges, df):
    """
    Adjusts audio feature target ranges based on the user's tester song feedback.
    
    Args:
        accepted_tracks (list): Track names the user accepted
        rejected_tracks (list): Track names the user rejected
        current_ranges (dict): Current audio feature target ranges
        df (DataFrame): The dataset used for the search
    Returns:
        dict: Updated audio feature ranges
    """
    updated_ranges = current_ranges.copy()

    # Get audio features of rejected tracks and tighten ranges away from them
    if rejected_tracks:
        rejected_df = df[df["track_name"].isin(rejected_tracks)]
        for feature in ["energy", "valence", "danceability", "tempo"]:
            if feature in rejected_df.columns and feature in updated_ranges:
                rejected_mean = rejected_df[feature].mean()
                current_min = updated_ranges[feature]["min"]
                current_max = updated_ranges[feature]["max"]
                midpoint = (current_min + current_max) / 2
                # Push range away from rejected mean
                if rejected_mean < midpoint:
                    updated_ranges[feature]["min"] = min(current_max, current_min + 0.1)
                else:
                    updated_ranges[feature]["max"] = max(current_min, current_max - 0.1)

    return updated_ranges

In [0]:
# Test the tool
current_ranges = {
    "energy": {"min": 0.8, "max": 1.0},
    "valence": {"min": 0.5, "max": 1.0},
    "danceability": {"min": 0.7, "max": 1.0},
    "tempo": {"min": 120, "max": 150}
}

df = spark.table("main.vibeml.vibeml_tracks_processed").toPandas()

accepted = ["Yakuza"]
rejected = ["Touch Me"]

updated = feedback_refinement(accepted, rejected, current_ranges, df)
print("Updated ranges after feedback:")
print(json.dumps(updated, indent=2))

### Tool 8 - Save Playlist: save_playlist
- Saves the final list of tracks as a new playlist to the user's Spotify account.

In [0]:
def save_playlist(sp, track_ids, playlist_name):
    """
    Saves the final list of tracks as a new playlist to the user's Spotify account.
    
    Args:
        sp: Authenticated Spotipy client
        track_ids (list): List of Spotify track IDs to add to the playlist
        playlist_name (str): Name of the playlist to create
    Returns:
        str: URL of the created playlist
    """
    pass

### Tool 9: Constraint Filter: constraint_filter

Filters songs from the dataset based on the user's specific preferences or limits such as explicit content, song duration, or popularity.

In [0]:
def constraint_filter(df, explicit=None, max_duration_ms=None, min_popularity=None):
    """
    Filters songs based on the user's specific preferences or limits.
    
    Args:
        df (DataFrame): The dataset to filter
        explicit (bool): If False, exclude explicit tracks. If True, include only explicit tracks. If None, no filter.
        max_duration_ms (int): Maximum song duration in milliseconds
        min_popularity (int): Minimum popularity score between 0 and 100
    Returns:
        DataFrame: Filtered tracks based on the specified constraints
    """
    filtered = df.copy()

    if explicit is not None:
        filtered = filtered[filtered["explicit"] == explicit]
    
    if max_duration_ms is not None:
        filtered = filtered[filtered["duration_ms"] <= max_duration_ms]
    
    if min_popularity is not None:
        filtered = filtered[filtered["popularity"] >= min_popularity]
    
    return filtered.reset_index(drop=True)

# Test the tool
df = spark.table("main.vibeml.vibeml_tracks_processed").toPandas()

filtered = constraint_filter(df, explicit=False, max_duration_ms=300000, min_popularity=50)
print(f"Total tracks before filtering: {len(df)}")
print(f"Total tracks after filtering: {len(filtered)}")
print(filtered[["track_name", "artists", "explicit", "duration_ms", "popularity"]].head(5))

### Tool 10 - Diversity Tool: deduplication

Prevents the playlist from becoming repetitive by ensuring no artist appears more than once and removing duplicate tracks.

In [0]:
def deduplication(df, max_per_artist=1):
    """
    Prevents the playlist from becoming repetitive by limiting how many times
    an artist can appear and removing duplicate tracks.
    
    Args:
        df (DataFrame): The dataset to deduplicate
        max_per_artist (int): Maximum number of tracks allowed per artist
    Returns:
        DataFrame: Deduplicated tracks with artist diversity enforced
    """
    # Remove duplicate track names
    df = df.drop_duplicates(subset=["track_name"]).reset_index(drop=True)
    
    # Limit tracks per artist
    df = df.groupby("artists").head(max_per_artist).reset_index(drop=True)
    
    return df

In [0]:
# Test the tool
df = spark.table("main.vibeml.vibeml_tracks_processed").toPandas()

# First run a song search to get a subset
results = search_songs(energy=(0.8, 1.0), danceability=(0.7, 1.0), limit=50)
print(f"Tracks before deduplication: {len(results)}")
print(f"Unique artists before: {results['artists'].nunique()}")

deduped = deduplication(results, max_per_artist=1)
print(f"Tracks after deduplication: {len(deduped)}")
print(f"Unique artists after: {deduped['artists'].nunique()}")
print(deduped[["track_name", "artists"]].head(10))

## Agent Definition

This section defines the agent's system prompt, registers the tools, and sets up the LLM. The agent uses a ReAct style reasoning pattern, that is it reasons over the user's input, decides which tools to call, observes the results, and reasons again until it has enough information to build the final playlist.

In [0]:
# Tool list - full implementation will be added in the next phase
TOOLS = [
    get_user_profile,
    get_currently_playing,
    search_catalog,
    search_songs,
    trend_filter,
    vibe_mapping,
    feedback_refinement,
    save_playlist
]

print("Agent definition loaded successfully")

## Test Run
A simple test to verify the agent setup is in place and the tools are registered correctly. Full end to end testing will be completed once the LLM and Spotify API authentication are configured.


In [0]:
# Test that all tools are defined and accessible
def test_agent_setup():
    print("Testing agent setup...\n")
    
    # Check all tools are defined
    for tool in TOOLS:
        print(f"Tool registered: {tool.__name__}")
    
    # Check system prompt is loaded
    print("\nSystem prompt loaded successfully")
    print(f"Total tools registered: {len(TOOLS)}")
    print("\nAgent setup test complete.")

test_agent_setup()